## Triple

In [ ]:
from openai import OpenAI
import pandas as pd
import json
import time
from tqdm import tqdm
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed

client = OpenAI(api_key="your api key",
                base_url="https://")

def get_triplets_from_ai(chunk_text):
    """Call API to extract triplets in English"""
    system_prompt = """
    You are a medical knowledge engineer and endocrinology expert specializing in the construction of structured diabetes knowledge graphs. Your core responsibility is to translate authoritative medical information into standardized knowledge triples. These triples serve as the standard for cross-referencing and validating information extracted from patient-facing sources.
    Requirements:
    1.Format: must be a standard json list of objects: [("S": "Subject", "P": "Predicate", "O": "Object"}].
    2.Predicate (P) should be professional and standardized, such as: treats, causes, symptom of, prevents, belongs to, contraindicated for, increases risk of, lowers blood glucose, associated with, and so on
    3.Language: The output must be in English.
    4.If no clear triplets are found, return an empty list [].

    Example:
    Content: “Fosinopril contraindication diabetes mellitus.”
    Triple Extraction: 
    "S": "Fosinopril",
    "P": "contraindicated for",
    "O": "Diabetes mellitus"

    You need to generate 3 independent reasoning paths for the same video content, then select the most consistent result through majority vote:
    For triple extraction: If 2 or 3 paths generate the same standardized triple, adopt that triple; if there are differences, prioritize the triple that matches authoritative terminology.
    """

    # Few-shot Example in English
    user_example = "Text: 'Metformin can effectively lower blood glucose levels in patients with Type 2 Diabetes.'\nOutput:"
    assistant_example = '[{"S": "Metformin", "P": "lowers_blood_glucose", "O": "Type 2 Diabetes"}]'

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_example},
                {"role": "assistant", "content": assistant_example},
                {"role": "user", "content": f"Text to process: '{chunk_text}'\nOutput:"}
            ],
            response_format={ "type": "json_object" }
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error during API call: {e}")
        return "[]"

In [2]:
df = pd.read_csv("dataset/allknowledge.csv", encoding="utf-8")

# 检查 'text' 列是否存在
if "text" not in df.columns:
    raise ValueError("The column named 'text' was not found in the CSV file. Please check the column name.")

print(f"A total of {len(df)} pieces of text were read.")

A total of 5281 pieces of text were read.


In [6]:
# Old version 旧版本 时间过长
# triplets_list = []

# for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting triplets"):
#     text = row["text"]
#     if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
#         triplets_list.append("[]")  # 空文本返回空列表
#         continue
    
#     result_str = get_triplets_from_ai(text)
#     # 尝试解析一下，确保返回的是合法 JSON（可选）
#     try:
#         # 验证是否能解析为 JSON，如果失败则存储空列表
#         json.loads(result_str)
#         triplets_list.append(result_str)
#     except json.JSONDecodeError:
#         print(f"Warning: The {idx}th item returned is not a valid JSON and has been emptied. Content: {result_str[:100]}")
#         triplets_list.append("[]")
    
#     time.sleep(0.5)  # 控制请求频率

# df["triplets_json"] = triplets_list

In [3]:
triplets_list = ["[]"] * len(df)

def process_row(idx, row):
    text = row["text"]
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return idx, "[]"
    
    result_str = get_triplets_from_ai(text)

    try:
        json.loads(result_str)
        result = result_str
    except json.JSONDecodeError:
        print(f"Warning: The {idx}th item returned is not a valid JSON and has been emptied. Content: {result_str[:100]}")
        result = "[]"
    
    time.sleep(0.1)   # 原来是 0.5，这里调小
    return idx, result

max_workers = 8   # 建议先从 5 开始，如果稳定再试 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_row, idx, row) for idx, row in df.iterrows()]
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting triplets"):
        idx, result = future.result()
        triplets_list[idx] = result

df["triplets_json"] = triplets_list

Extracting triplets:   3%|▎         | 170/5281 [03:23<1:22:46,  1.03it/s]

  "reasoning_paths": [
    [
      {
        "S": "Histamine",
        "P": "contraindicated for",


Extracting triplets: 100%|██████████| 5281/5281 [2:08:26<00:00,  1.46s/it]  


In [ ]:
output_file = "output_data/textknowledge_with_triplets.json"
df.to_json(output_file, orient="records", force_ascii=False, indent=2)
print(f"The result has been saved to {output_file}")

The result has been saved to output_data/textknowledge_with_triplets_ver4.json


In [ ]:
# expanded_rows = []
# for idx, row in df.iterrows():
#     try:
#         triplets = json.loads(row["triplets_json"])
#         for t in triplets:
#             expanded_rows.append({
#                 "source_row": idx,
#                 "subject": t.get("S", ""),
#                 "predicate": t.get("P", ""),
#                 "object": t.get("O", "")
#             })
#     except:
#         # 如果某行 JSON 解析失败，跳过
#         continue

# expanded_df = pd.DataFrame(expanded_rows)
# # expanded_df.to_csv("expanded_triplets.csv", index=False, encoding="utf-8")
# expanded_df.to_json(
#     "output_data/knowledge_triplets_ver4.json",
#     orient="records",
#     force_ascii=False,
#     indent=2
# )
# print("The expanded triples have been saved.")

The expanded triples have been saved.


In [ ]:
import pandas as pd
import json

# 读取原始文件
df = pd.read_json("output_data/textknowledge_with_triplets.json")

expanded_rows = []

# 优先提取这些“最终结果”字段
final_keys = [
    "majority_vote_result",
    "selected_triples",
    "final_output",
    "output",
    "final_triples"
]

for idx, row in df.iterrows():
    raw = row.get("triplets_json", "")

    if pd.isna(raw) or not isinstance(raw, str) or raw.strip() == "":
        continue

    try:
        obj = json.loads(raw)
    except Exception:
        continue

    triplets = []

    # 情况1：triplets_json 本身就是标准 list
    if isinstance(obj, list):
        for t in obj:
            if isinstance(t, dict) and {"S", "P", "O"}.issubset(t.keys()):
                triplets.append(t)

    # 情况2：triplets_json 是 dict，优先从最终结果字段里提取
    elif isinstance(obj, dict):
        for key in final_keys:
            value = obj.get(key)
            if isinstance(value, list):
                for t in value:
                    if isinstance(t, dict) and {"S", "P", "O"}.issubset(t.keys()):
                        triplets.append(t)

    # 如果同一行有重复三元组，先去重
    unique_triplets = []
    seen = set()

    for t in triplets:
        triple_key = (
            t.get("S", "").strip(),
            t.get("P", "").strip(),
            t.get("O", "").strip()
        )
        if triple_key not in seen:
            seen.add(triple_key)
            unique_triplets.append(t)

    # 展开成你要的格式
    for t in unique_triplets:
        expanded_rows.append({
            "source_row": idx,
            "subject": t.get("S", ""),
            "predicate": t.get("P", ""),
            "object": t.get("O", "")
        })

expanded_df = pd.DataFrame(expanded_rows)

# 输出为你要的 json 样式
expanded_df.to_json(
    "output_data/knowledge_triplets.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("The expanded triples have been saved.")
print("Original rows:", len(df))
print("Expanded triples:", len(expanded_df))
print(expanded_df.head())

The expanded triples have been saved.
Original rows: 5281
Expanded triples: 6012
   source_row             subject            predicate             object
0           0          Fosinopril  contraindicated for  Diabetes mellitus
1           1        Pyrazinamide  contraindicated for  Diabetes mellitus
2           2        Nicotinamide  contraindicated for  Diabetes mellitus
3           3  Estradiol valerate  contraindicated for  Diabetes mellitus
4           4          Disulfiram  contraindicated for  Diabetes mellitus


## Clean

In [ ]:
import pandas as pd
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.strip()
    
    # 去多余空格
    text = re.sub(r"\s+", " ", text)
    
    # 去掉首尾标点
    text = text.strip(".,;:!?")
    
    # 统一小写
    text = text.lower()
    
    return text

cleaned_rows = []

for _, row in expanded_df.iterrows():
    cleaned_rows.append({
        "source_row": row["source_row"],
        "subject": clean_text(row["subject"]),
        "predicate": clean_text(row["predicate"]),
        "object": clean_text(row["object"])
    })

cleaned_df = pd.DataFrame(cleaned_rows)

# ✅ 去重（核心步骤）
cleaned_df = cleaned_df.drop_duplicates(
    subset=["subject", "predicate", "object"]
).reset_index(drop=True)

# 输出
cleaned_df.to_json(
    "output_data/knowledge_triplets_clean.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("Cleaned triples saved.")
print("After cleaning:", len(cleaned_df))

Cleaned triples saved.
After cleaning: 4940
